# EN4554 – Assignment 2 (CNNs):

**Name:** `Dulsara G.M.L.`  **Index No.:** `220146A`  
**GitHub:** `https://github.com/lahirunie-dulsara/EN4554-Assignment-2-on-CNNs`

Task 1: Backpropagation

In [ ]:
# ============ COMMON SETUP (identical in every notebook) ============
import os, time, math, random, copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SMOKE = bool(int(os.environ.get('SMOKE', '0')))   # 1 -> tiny synthetic data, only to check the code runs
SEED = 0

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_all()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('device:', device, '| torch', torch.__version__, '| SMOKE =', SMOKE)

def load_cifar10(root='./data'):
    """45k train / 5k val split (fixed seed). Per-channel normalisation uses TRAINING statistics only.
    The official test set is never touched here (it is only loaded in Task 4)."""
    if SMOKE:
        g = torch.Generator().manual_seed(0)
        X = torch.rand(700, 3, 32, 32, generator=g); y = torch.randint(0, 10, (700,), generator=g); ntr = 600
    else:
        import torchvision
        ds = torchvision.datasets.CIFAR10(root, train=True, download=True)
        X = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float() / 255.
        y = torch.tensor(ds.targets); ntr = 45000
    perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
    tr, va = perm[:ntr], perm[ntr:]
    mean = X[tr].mean((0, 2, 3), keepdim=True); std = X[tr].std((0, 2, 3), keepdim=True)
    Xn = (X - mean) / std
    return Xn[tr].to(device), y[tr].to(device), Xn[va].to(device), y[va].to(device), mean, std

Xtr, ytr, Xva, yva, MEAN, STD = load_cifar10()
print('train', tuple(Xtr.shape), 'val', tuple(Xva.shape))

# ---------------- data augmentation (all implemented by hand, on GPU, per-sample randomness) ----------------
def random_flip(x):
    m = torch.rand(x.size(0), device=x.device) < 0.5
    return torch.where(m[:, None, None, None], x.flip(3), x)

def random_crop(x, pad=4):
    B, C, H, W = x.shape
    xp = F.pad(x, (pad,) * 4)                      # zero padding == mean colour after normalisation
    i = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    j = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]
    cols = (j[:, None] + ar)[:, None, None, :]
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows, cols]

def cutout(x, size=16):
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B,), device=x.device); cx = torch.randint(0, W, (B,), device=x.device)
    ys = torch.arange(H, device=x.device)[None, :, None]; xs = torch.arange(W, device=x.device)[None, None, :]
    y0 = (cy - size // 2)[:, None, None]; x0 = (cx - size // 2)[:, None, None]
    mask = (ys >= y0) & (ys < y0 + size) & (xs >= x0) & (xs < x0 + size)   # square clipped at the border
    return x.masked_fill(mask[:, None], 0.)

def mixup(x, y, alpha=0.4):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

AUGS = {'flip': random_flip, 'crop': random_crop, 'cutout': cutout}

# ---------------- training / evaluation ----------------
@torch.no_grad()
def evaluate(model, X, y, bs=1000):
    model.eval(); loss = 0.; correct = 0
    for k in range(0, len(X), bs):
        out = model(X[k:k + bs]).float()
        loss += F.cross_entropy(out, y[k:k + bs], reduction='sum').item()
        correct += (out.argmax(1) == y[k:k + bs]).sum().item()
    return loss / len(X), correct / len(X)

def cosine_warmup(opt, total_steps, warmup_steps=0):
    def f(s):
        if s < warmup_steps: return (s + 1) / warmup_steps
        p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train_model(model, opt, epochs, Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, bs=128, augs=(), mixup_alpha=0.,
                label_smoothing=0., sched=None, amp=False, eval_train=False, verbose=True):
    """Generic loop. 'train_loss' is the running mean of the (augmented) minibatch loss;
    'train_acc' (if eval_train) is measured in eval mode on the CLEAN training set."""
    model.to(device)
    use_amp = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    h = dict(train_loss=[], train_acc=[], val_loss=[], val_acc=[], diverged=False)
    n = len(Xtr)
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.; seen = 0
        perm = torch.randperm(n, device=device)
        for k in range(0, n - bs + 1, bs):
            idx = perm[k:k + bs]; x, y = Xtr[idx], ytr[idx]
            for a in augs: x = AUGS[a](x)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                if mixup_alpha > 0:
                    x, ya, yb, lam = mixup(x, y, mixup_alpha); out = model(x)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing) + \
                           (1 - lam) * F.cross_entropy(out, yb, label_smoothing=label_smoothing)
                else:
                    out = model(x); loss = F.cross_entropy(out, y, label_smoothing=label_smoothing)
            if not torch.isfinite(loss):
                h['diverged'] = True; break
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            run += loss.item() * bs; seen += bs
        if h['diverged']:
            print(f'  diverged at epoch {ep+1}'); break
        h['train_loss'].append(run / seen)
        vl, va = evaluate(model, Xva, yva); h['val_loss'].append(vl); h['val_acc'].append(va)
        if eval_train: h['train_acc'].append(evaluate(model, Xtr, ytr)[1])
        if verbose:
            print(f'  ep {ep+1:3d}/{epochs} train_loss {h["train_loss"][-1]:.4f} val_loss {vl:.4f} '
                  f'val_acc {va:.4f}' + (f' train_acc {h["train_acc"][-1]:.4f}' if eval_train else '') +
                  f' ({time.time()-t0:.1f}s)')
    return h

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# ---------------- models ----------------
class SmallCNN(nn.Module):
    """3 x [conv3x3 - (BN) - ReLU - maxpool - (dropout)] followed by a 2-layer classifier."""
    def __init__(self, bn=False, dropout=False, widths=(32, 64, 128)):
        super().__init__()
        L = []; cin = 3
        for w in widths:
            L += [nn.Conv2d(cin, w, 3, padding=1, bias=not bn)]
            if bn: L += [nn.BatchNorm2d(w)]
            L += [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            if dropout: L += [nn.Dropout(0.25)]
            cin = w
        self.features = nn.Sequential(*L)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(widths[-1] * 16, 256), nn.ReLU(inplace=True),
                                        nn.Dropout(0.5) if dropout else nn.Identity(), nn.Linear(256, 10))
    def forward(self, x): return self.classifier(self.features(x))

class Block(nn.Module):
    """Two 3x3 conv-BN layers. residual=False removes ONLY the shortcut (plain network)."""
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.residual = residual
        self.short = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
                      if residual and (stride != 1 or cin != cout) else nn.Identity())
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual: out = out + self.short(x)
        return F.relu(out)

class CifarNet(nn.Module):
    """CIFAR-style net of depth 6n+2: conv3x3 -> 3 stages x n blocks -> global avg pool -> linear."""
    def __init__(self, n, residual=True, widths=(16, 32, 64), num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        blocks = []; cin = widths[0]
        for s, w in enumerate(widths):
            for b in range(n):
                blocks.append(Block(cin, w, 2 if (b == 0 and s > 0) else 1, residual)); cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(widths[-1], num_classes)
        for m in self.modules():                      # Kaiming initialisation
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear): nn.init.kaiming_normal_(m.weight, nonlinearity='relu'); nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))
